# NB16 — LLM Zero-/Few-Shot Baselines (OpenRouter)

**Run location:** LOCAL / any CPU machine. **No GPU needed** — this notebook only calls an API.

## Why this notebook exists
As of 2025–2026, the single most predictable reviewer objection to a low-resource sarcasm paper is:
*"Why is there no LLM baseline?"* This notebook closes that gap. It is **standalone**: it re-runs
nothing from NB01–NB15 and only *reads* existing artefacts from `04_outputs/`.

## What it does
1. Loads the leakage-controlled test splits produced by NB01 (never re-splits anything).
2. Queries one or more OpenRouter models **zero-shot** and **few-shot** on all four task variants.
3. Optionally runs a **few-shot cross-corpus 3x3 matrix** with the same leakage blocking NB10 uses.
4. Scores everything with macro-F1 + bootstrap CIs, and runs **McNemar** against the fine-tuned
   proposed model on the shared test items.

## Where it writes
| Path | Contents |
|---|---|
| `04_outputs/llm_calls/cache/*.jsonl` | Response cache (resume-safe; re-runs cost nothing) |
| `04_outputs/llm_calls/predictions/*.csv` | Per-run predictions, NB12-compatible schema |
| `04_outputs/llm_calls/run_config.json`, `cost_report.json` | Provenance + spend |
| `04_outputs/finalized_outputs/tables/18_*.csv` | Paper tables |
| `04_outputs/finalized_outputs/figures/F_llm_*.{pdf,png}` | Paper figures (NB15 style) |

**It does not touch anything NB15 wrote.** You do not need to re-run NB15.

## Required `.env` (in repo root)
```
OPENROUTER_API_KEY=sk-or-...
OPENROUTER_BASE_URL=https://openrouter.ai/api/v1
OPENROUTER_MODEL=openai/gpt-4o-mini,meta-llama/llama-3.1-8b-instruct
NORMALIZER_PROVIDER=openai/gpt-4o-mini
```
- `OPENROUTER_MODEL` accepts a **comma-separated list** — every model is evaluated.
- `NORMALIZER_PROVIDER` is the model used to rescue responses that regex cannot parse.
  Set it to `none` to disable and rely on regex only.

## Runtime
- `DEBUG=True`: ~1–2 min (32 items/task, 1 model) — **always run this first**.
- `DEBUG=False`, `MAX_EVAL_N=500`, 2 models x 2 settings: ~15–30 min, roughly 8k calls.
- `DEBUG=False`, `MAX_EVAL_N=None` (full test sets), 2 models x 2 settings: ~1–2 h, roughly 22k calls.

Cost with cheap models (gpt-4o-mini / llama-3.1-8b) is typically **well under $5** for the full run.
The cost estimate is printed before any spending happens.


## 0 · Dependencies

In [ ]:
# Pinned to prevent silent breaking changes mid-project. `requests` is all we need for
# OpenRouter — no vendor SDK, so the notebook keeps working if their client library churns.
%pip install requests==2.32.3 python-dotenv==1.0.1 --quiet
%pip install pandas==2.2.2 numpy==1.26.4 scikit-learn==1.5.1 scipy==1.14.0 --quiet
%pip install matplotlib==3.9.2 --quiet
print("deps installed")

In [ ]:
# Version verification — catches the case where a pinned install silently failed and an
# incompatible pre-existing version is being imported instead.
import requests, pandas, numpy, sklearn, scipy, matplotlib
print(f"requests    : {requests.__version__}")
print(f"pandas      : {pandas.__version__}")
print(f"numpy       : {numpy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"scipy       : {scipy.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")

## 1 · Config

`DEBUG=True` runs a 32-item smoke test per task on a single model. **Run it first.** It exercises
every code path (auth, caching, parsing, scoring, figures) for a few cents.

In [ ]:
# === DEBUG MODE ===
# Set DEBUG = True to run a fast smoke test on a tiny subset before the full run.
# Always run DEBUG = True first to catch errors cheaply, then set to False for the full run.
DEBUG = False                # <-- change to False for the full run

DEBUG_SAMPLES   = 32        # items per task in debug mode
DEBUG_MODELS    = 1         # only the first model in OPENROUTER_MODEL during debug

# === EVALUATION SCOPE ===
# MAX_EVAL_N caps items per task to control spend. Set to None to use the FULL test split
# (preferred for the camera-ready — reviewers dislike subsampled baselines, and the full run
# is still cheap on small models). Sampling is stratified + seeded, so it is reproducible.
MAX_EVAL_N = None

SEED = 42

# Task variants to evaluate. These names match NB01's split filenames exactly.
TASKS = [
    "ben_sarc_binary",
    "banglasarc_binary",
    "banglasarc3_binary",
    "banglasarc3_ternary",
]

# === PROMPTING ===
SETTINGS        = ["zero_shot", "few_shot"]
PROMPT_LANG     = "en"      # "en" = English instructions + Bangla text (standard, usually strongest)
                            # "bn" = fully Bangla instructions. Flip this for a cheap prompt-language
                            #        ablation if a reviewer asks whether English instructions
                            #        disadvantage the LLM.
FEWSHOT_PER_CLASS = 3       # balanced exemplars per class: binary -> 6-shot, ternary -> 9-shot.
                            # Balanced beats "exactly 5" here because an unbalanced exemplar set
                            # biases the LLM's prior and confounds the comparison.

# === CROSS-CORPUS (few-shot only) ===
# Zero-shot has no source corpus, so a transfer matrix is meaningless for it. Few-shot does have
# one (the exemplars), so a 3x3 matrix is well-defined and directly parallels NB10.
RUN_CROSS_CORPUS = True
CROSS_CORPUS_MAX_N = 300 if not DEBUG else 16
CROSS_CORPORA = ["ben_sarc", "banglasarc", "banglasarc3"]

# === API BEHAVIOUR ===
MAX_WORKERS     = 8         # concurrent requests; lower to 2-4 if you hit sustained 429s
MAX_RETRIES     = 5
TIMEOUT_S       = 90
TEMPERATURE     = 0.0       # deterministic: this is a measurement, not a generation task
MAX_TOKENS      = 8         # we want one label word; capping this also caps cost

# === METRICS ===
BOOTSTRAP_B     = 1000

# Unparseable/refused responses are assigned the training-majority class rather than dropped.
# Dropping them would silently flatter the LLM by letting it skip hard items. We report both
# the fallback-inclusive score (headline) and the excluded-failures score (for transparency).
FALLBACK_TO_MAJORITY = True

print(f"DEBUG={DEBUG} | MAX_EVAL_N={MAX_EVAL_N} | tasks={len(TASKS)} | settings={SETTINGS}")
print(f"cross-corpus={RUN_CROSS_CORPUS} | prompt_lang={PROMPT_LANG}")

## 2 · Paths & environment

In [ ]:
import os, re, json, time, math, random, hashlib, threading, unicodedata, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
warnings.filterwarnings("ignore")

random.seed(SEED); np.random.seed(SEED)

# Same root-resolution strategy as NB10/NB12/NB15 so this notebook drops into the repo unchanged.
def find_repo_root():
    cands = [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents)
    for c in cands:
        if (c / "01_data").exists() or (c / "04_outputs").exists():
            return c.resolve()
    raise RuntimeError("repo root not found (need a dir containing 01_data/ or 04_outputs/).")

ROOT   = find_repo_root()
OUT    = ROOT / "04_outputs"
SPLITS = ROOT / "01_data" / "interim" / "splits"
TABLES = OUT / "tables"
PRED   = OUT / "predictions"

# NB16's own workspace — nothing here collides with NB01-15 outputs.
LLM         = OUT / "llm_calls"
LLM_CACHE   = LLM / "cache"
LLM_PRED    = LLM / "predictions"
LLM_RAW     = LLM / "raw"

# Finalized paper artefacts land beside NB15's, so you never re-run NB15.
FINAL = OUT / "finalized_outputs"
FT, FF = FINAL / "tables", FINAL / "figures"

for p in (LLM, LLM_CACHE, LLM_PRED, LLM_RAW, FT, FF):
    p.mkdir(parents=True, exist_ok=True)

print("ROOT  :", ROOT)
print("SPLITS:", SPLITS, "->", "OK" if SPLITS.exists() else "MISSING (run NB01 first)")
print("LLM   :", LLM)
print("FINAL :", FINAL)

In [ ]:
# Load .env from the repo root. We do NOT hardcode or echo the key anywhere.
from dotenv import load_dotenv
env_path = ROOT / ".env"
load_dotenv(env_path)
print(f".env: {env_path} ->", "found" if env_path.exists() else "NOT FOUND (using shell env)")

API_KEY  = os.getenv("OPENROUTER_API_KEY", "").strip()
BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1").strip().rstrip("/")
MODEL_ENV = os.getenv("OPENROUTER_MODEL", "").strip()
NORMALIZER = os.getenv("NORMALIZER_PROVIDER", "").strip()

# Comma-separated model list -> evaluate each. Two models (one frontier, one open-weight) is the
# sweet spot: it shows the finding is not an artefact of a single vendor without doubling cost twice.
MODELS = [m.strip() for m in MODEL_ENV.split(",") if m.strip()]
if DEBUG:
    MODELS = MODELS[:DEBUG_MODELS]

USE_NORMALIZER = NORMALIZER.lower() not in ("", "none", "regex", "off", "false")

assert API_KEY,  "OPENROUTER_API_KEY missing — add it to .env"
assert MODELS,   "OPENROUTER_MODEL missing — add e.g. OPENROUTER_MODEL=openai/gpt-4o-mini to .env"

print("base_url  :", BASE_URL)
print("models    :", MODELS)
print("normalizer:", NORMALIZER if USE_NORMALIZER else "(disabled — regex parsing only)")
print("api_key   :", API_KEY[:8] + "..." + API_KEY[-4:])

## 3 · Load splits (read-only)

We read NB01's splits directly. Nothing is re-split, re-cleaned, or re-deduplicated — the
leakage-controlled splits are treated as immutable inputs.

In [ ]:
# NB01 wrote splits as {variant}_{train,val,test}.csv. rglob fallback mirrors NB15's locator so
# this still works if the repo layout shifted.
def find_split(variant, split):
    p = SPLITS / f"{variant}_{split}.csv"
    if p.exists():
        return p
    for q in ROOT.rglob(f"{variant}_{split}.csv"):
        if "finalized_outputs" not in q.parts:
            return q
    return None

def label_col(df):
    for c in ("label_binary", "label_ternary", "label", "gold_label"):
        if c in df.columns:
            return c
    raise KeyError(f"no label column in {list(df.columns)}")

_ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\ufeff"), None)
def norm_key(s):
    # Identical to NB01/NB10 norm_key so leakage sets line up exactly across notebooks.
    if not isinstance(s, str):
        s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    return re.sub(r"\s+", " ", s).strip().casefold()

def load_split(variant, split):
    p = find_split(variant, split)
    if p is None:
        return None
    df = pd.read_csv(p)
    df = df.dropna(subset=["text"]).copy()
    df["text"] = df["text"].astype(str)
    lc = label_col(df)
    df["y"] = df[lc].astype(int)
    if "norm_key" not in df.columns:
        df["norm_key"] = df["text"].map(norm_key)
    return df[["text", "y", "norm_key"]].reset_index(drop=True)

# Label vocabularies — must match NB01's BIN_MAP / TERN_MAP exactly or every score is wrong.
BIN_LABELS  = {0: "Non-Sarcastic", 1: "Sarcastic"}
TERN_LABELS = {0: "Non-Sarcastic", 1: "Neutral", 2: "Sarcastic"}
def labels_for(task):
    return TERN_LABELS if task.endswith("ternary") else BIN_LABELS

DATA = {}
for t in TASKS:
    tr, te = load_split(t, "train"), load_split(t, "test")
    if tr is None or te is None:
        print(f"  SKIP {t:22s} (splits not found)")
        continue
    DATA[t] = {"train": tr, "test": te}
    print(f"  ok   {t:22s} train={len(tr):6d} test={len(te):5d} classes={sorted(te['y'].unique())}")

assert DATA, "No task splits found — run NB01 first."

In [ ]:
# Stratified, seeded subsample. Stratification keeps the class balance identical to the full test
# split, so the subsampled macro-F1 stays an unbiased estimate of the full-split score.
def subsample(df, n, seed=SEED):
    if n is None or len(df) <= n:
        return df.reset_index(drop=True)
    parts = []
    for cls, grp in df.groupby("y"):
        k = max(1, int(round(n * len(grp) / len(df))))
        parts.append(grp.sample(min(k, len(grp)), random_state=seed))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

EVAL = {}
for t, d in DATA.items():
    EVAL[t] = subsample(d["test"], MAX_EVAL_N)
    frac = len(EVAL[t]) / len(d["test"])
    print(f"  {t:22s} eval n={len(EVAL[t]):5d} / {len(d['test']):5d}  ({frac:.0%} of test)")

# Majority class from TRAIN (never test) — used as the fallback for unparseable responses.
MAJORITY = {t: int(DATA[t]["train"]["y"].value_counts().idxmax()) for t in DATA}
print("\ntrain-majority fallback class per task:",
      {t: labels_for(t)[c] for t, c in MAJORITY.items()})

## 4 · Prompts

One instruction template, two settings. Few-shot exemplars are drawn **only from the train split**
— using test items as exemplars would be exactly the leakage this paper is about.

In [ ]:
SYS_EN = (
    "You are an expert annotator of Bengali (Bangla) social media comments. "
    "You classify whether a comment is sarcastic. "
    "Answer with exactly one label and nothing else — no explanation, no punctuation."
)
SYS_BN = (
    "আপনি বাংলা সোশ্যাল মিডিয়া মন্তব্যের একজন বিশেষজ্ঞ annotator। "
    "আপনি নির্ধারণ করেন একটি মন্তব্য ব্যঙ্গাত্মক (sarcastic) কি না। "
    "শুধুমাত্র একটি লেবেল দিয়ে উত্তর দিন, অন্য কিছু নয়।"
)

TASK_EN = {
    "binary":  ("Classify the comment as one of: SARCASTIC, NON_SARCASTIC.\n"
                "SARCASTIC = the literal meaning differs from the intended meaning, typically "
                "mocking, ironic, or feigning praise.\n"
                "NON_SARCASTIC = the comment means what it literally says."),
    "ternary": ("Classify the comment as one of: SARCASTIC, NON_SARCASTIC, NEUTRAL.\n"
                "SARCASTIC = the literal meaning differs from the intended meaning (mocking/ironic).\n"
                "NON_SARCASTIC = the comment means what it literally says and carries clear sentiment.\n"
                "NEUTRAL = plain, factual, or emotionless with no sarcastic intent."),
}
TASK_BN = {
    "binary":  ("মন্তব্যটিকে এই দুটির একটিতে শ্রেণিবদ্ধ করুন: SARCASTIC, NON_SARCASTIC।\n"
                "SARCASTIC = আক্ষরিক অর্থ ও উদ্দিষ্ট অর্থ ভিন্ন — ব্যঙ্গ, শ্লেষ বা ভান করা প্রশংসা।\n"
                "NON_SARCASTIC = মন্তব্যটি আক্ষরিক অর্থেই যা বলছে তাই বোঝায়।"),
    "ternary": ("মন্তব্যটিকে এই তিনটির একটিতে শ্রেণিবদ্ধ করুন: SARCASTIC, NON_SARCASTIC, NEUTRAL।\n"
                "SARCASTIC = আক্ষরিক অর্থ ও উদ্দিষ্ট অর্থ ভিন্ন (ব্যঙ্গ/শ্লেষ)।\n"
                "NON_SARCASTIC = আক্ষরিক অর্থেই যা বলছে তাই বোঝায়, স্পষ্ট অনুভূতিসহ।\n"
                "NEUTRAL = নিরপেক্ষ, তথ্যভিত্তিক, ব্যঙ্গের উদ্দেশ্য নেই।"),
}

# Canonical output tokens the model is asked to emit, per label id.
TOKENS_BIN  = {0: "NON_SARCASTIC", 1: "SARCASTIC"}
TOKENS_TERN = {0: "NON_SARCASTIC", 1: "NEUTRAL", 2: "SARCASTIC"}
def tokens_for(task):
    return TOKENS_TERN if task.endswith("ternary") else TOKENS_BIN

def kind(task):
    return "ternary" if task.endswith("ternary") else "binary"

def pick_exemplars(train_df, task, per_class=FEWSHOT_PER_CLASS, seed=SEED):
    # A single fixed exemplar set is reused for every test item in a run. Per-item retrieval would
    # confound the comparison (it becomes a retrieval system, not a few-shot LLM baseline) and
    # would make the run non-reproducible.
    toks = tokens_for(task)
    rows = []
    for cls in sorted(train_df["y"].unique()):
        grp = train_df[train_df["y"] == cls]
        # Prefer mid-length exemplars: very short comments carry no sarcasm signal, very long ones
        # eat the context budget for no benefit.
        grp = grp.assign(_L=grp["text"].str.len())
        grp = grp[(grp["_L"] >= 20) & (grp["_L"] <= 200)]
        if len(grp) < per_class:
            grp = train_df[train_df["y"] == cls]
        rows.append(grp.sample(min(per_class, len(grp)), random_state=seed))
    ex = pd.concat(rows).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return [(r["text"], toks[int(r["y"])]) for _, r in ex.iterrows()]

def build_messages(task, setting, text, exemplars=None):
    k = kind(task)
    sys_p  = SYS_EN if PROMPT_LANG == "en" else SYS_BN
    task_p = (TASK_EN if PROMPT_LANG == "en" else TASK_BN)[k]
    msgs = [{"role": "system", "content": sys_p}]
    user = task_p + "\n\n"
    if setting == "few_shot" and exemplars:
        # Exemplars go in the user turn as labelled examples rather than as fake assistant turns:
        # it is more robust across providers that handle multi-turn priming differently.
        user += "Examples:\n"
        for tx, lb in exemplars:
            user += f'Comment: {tx}\nLabel: {lb}\n\n'
    user += f"Now classify this comment.\nComment: {text}\nLabel:"
    msgs.append({"role": "user", "content": user})
    return msgs

# Sanity check the prompt before spending anything.
_t = TASKS[0] if TASKS[0] in DATA else list(DATA)[0]
_ex = pick_exemplars(DATA[_t]["train"], _t)
_m = build_messages(_t, "few_shot", DATA[_t]["test"].iloc[0]["text"], _ex)
print(f"--- sample few-shot prompt [{_t}] ---\n")
print(_m[0]["content"][:200], "\n")
print(_m[1]["content"][:900], "...")

## 5 · OpenRouter client — cached, retrying, concurrent

The cache is the checkpoint/resume mechanism. Every response is keyed by a hash of
(model, task, setting, prompt) and appended to a JSONL. **Re-running the notebook costs nothing**
for calls already made — interrupt it freely and re-run.

In [ ]:
_cache_lock = threading.Lock()
_stats_lock = threading.Lock()
STATS = {"hits": 0, "calls": 0, "errors": 0, "prompt_tokens": 0, "completion_tokens": 0, "cost_usd": 0.0}

def cache_path(model):
    # One JSONL per model keeps files small and lets you delete a single model's cache to redo it.
    slug = re.sub(r"[^A-Za-z0-9]+", "_", model).strip("_")
    return LLM_CACHE / f"{slug}.jsonl"

def req_hash(model, task, setting, messages):
    # Hash the full prompt, so any prompt edit correctly invalidates the cache instead of silently
    # serving a stale answer to a different question.
    blob = json.dumps({"m": model, "t": task, "s": setting, "msgs": messages,
                       "temp": TEMPERATURE, "lang": PROMPT_LANG},
                      ensure_ascii=False, sort_keys=True)
    return hashlib.sha1(blob.encode("utf-8")).hexdigest()

def load_cache(model):
    p = cache_path(model)
    if not p.exists():
        return {}
    out = {}
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
                out[r["hash"]] = r
            except Exception:
                continue  # a torn last line from a hard interrupt — safe to skip
    return out

def append_cache(model, rec):
    with _cache_lock:
        with open(cache_path(model), "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

CACHES = {m: load_cache(m) for m in MODELS}
for m in MODELS:
    print(f"  cache {m:45s} {len(CACHES[m]):6d} cached responses")

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    # OpenRouter asks for these for attribution; harmless and keeps you in good standing.
    "HTTP-Referer": "https://localhost/bangla-sarcasm-research",
    "X-Title": "Bangla Sarcasm LLM Baseline",
})

class RateLimited(Exception):
    pass

def _post(payload):
    return SESSION.post(f"{BASE_URL}/chat/completions", json=payload, timeout=TIMEOUT_S)

def call_llm(model, messages, max_tokens=MAX_TOKENS, temperature=TEMPERATURE):
    """One chat completion with exponential backoff + jitter. Returns (text, usage_dict).

    Retries on 429 and 5xx only — a 400/401 is a bug or a bad key and retrying just wastes time,
    so those raise immediately.
    """
    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "usage": {"include": True},   # OpenRouter returns real spend when this is set
    }
    delay = 2.0
    last = None
    for attempt in range(MAX_RETRIES):
        try:
            r = _post(payload)
            if r.status_code == 200:
                j = r.json()
                if "choices" not in j or not j["choices"]:
                    raise RuntimeError(f"no choices in response: {str(j)[:200]}")
                txt = (j["choices"][0]["message"].get("content") or "").strip()
                return txt, j.get("usage", {}) or {}
            if r.status_code == 429:
                # Honour Retry-After when the provider sends it — guessing is worse than obeying.
                ra = r.headers.get("Retry-After")
                wait = float(ra) if ra and ra.replace(".", "", 1).isdigit() else delay
                time.sleep(wait + random.uniform(0, 1)); delay = min(delay * 2, 60)
                last = RateLimited(f"429 (attempt {attempt+1})")
                continue
            if 500 <= r.status_code < 600:
                time.sleep(delay + random.uniform(0, 1)); delay = min(delay * 2, 60)
                last = RuntimeError(f"{r.status_code}: {r.text[:200]}")
                continue
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:300]}")
        except (requests.Timeout, requests.ConnectionError) as e:
            time.sleep(delay + random.uniform(0, 1)); delay = min(delay * 2, 60)
            last = e
    raise RuntimeError(f"failed after {MAX_RETRIES} retries: {last}")

# Smoke-test auth before launching thousands of calls — fail in 2 seconds, not 20 minutes.
try:
    _txt, _u = call_llm(MODELS[0], [{"role": "user", "content": "Reply with the single word: OK"}], max_tokens=5)
    print(f"auth OK -> {MODELS[0]} replied: {_txt!r}")
    print("usage:", _u)
except Exception as e:
    raise RuntimeError(f"OpenRouter smoke test failed — check key/base_url/model. {e}")

## 6 · Label parsing

Regex first (free, deterministic). Only responses regex cannot resolve go to the
`NORMALIZER_PROVIDER` model. Anything still unresolved is recorded as a parse failure and
assigned the train-majority class — never silently dropped.

In [ ]:
def parse_label(text, task):
    """Regex-parse a raw response into a label id, or None if ambiguous.

    Order matters: NON_SARCASTIC contains 'SARCASTIC' as a substring, so the negative and neutral
    classes must be tested before the positive one or every negative gets misread as positive.
    """
    if not text:
        return None
    t = text.strip().upper()
    t = re.sub(r"[^A-Z_ ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()

    is_tern = task.endswith("ternary")
    has_neutral = bool(re.search(r"\bNEUTRAL\b", t))
    has_non = bool(re.search(r"NON[ _]?SARCAS|NOT SARCAS|\bNONSARCAS", t))
    has_sarc = bool(re.search(r"\bSARCAS", t))

    if is_tern:
        if has_neutral and not (has_non or has_sarc): return 1
        if has_non:  return 0
        if has_neutral: return 1
        if has_sarc: return 2
    else:
        if has_non:  return 0
        if has_sarc: return 1
    return None

NORM_SYS = ("You map a messy classifier output to exactly one canonical label. "
            "Reply with only the label token, nothing else.")

# Normalizer responses are cached too, so a re-run is genuinely free and byte-identical.
NORM_CACHE = load_cache(NORMALIZER) if USE_NORMALIZER else {}
if USE_NORMALIZER:
    print(f"  cache {NORMALIZER:45s} {len(NORM_CACHE):6d} cached normalizer responses")

def normalize_label(text, task):
    """LLM fallback for responses regex cannot parse (refusals, hedges, translated labels)."""
    if not USE_NORMALIZER:
        return None
    opts = " / ".join(tokens_for(task).values())
    msgs = [{"role": "system", "content": NORM_SYS},
            {"role": "user", "content":
             f"Allowed labels: {opts}\n\nModel output:\n{text[:500]}\n\n"
             f"Which allowed label does this output mean? If it means none of them, reply UNKNOWN."}]
    h = req_hash(NORMALIZER, task, "normalize", msgs)
    if h in NORM_CACHE:
        with _stats_lock:
            STATS["hits"] += 1
        return parse_label(NORM_CACHE[h].get("raw", ""), task)
    try:
        out, u = call_llm(NORMALIZER, msgs, max_tokens=8)
        rec = {"hash": h, "model": NORMALIZER, "task": task, "setting": "normalize", "raw": out,
               "prompt_tokens": int(u.get("prompt_tokens", 0) or 0),
               "completion_tokens": int(u.get("completion_tokens", 0) or 0),
               "cost": float(u.get("cost", 0) or 0), "error": None}
        append_cache(NORMALIZER, rec)
        NORM_CACHE[h] = rec
        with _stats_lock:
            STATS["calls"] += 1
            STATS["prompt_tokens"] += rec["prompt_tokens"]
            STATS["completion_tokens"] += rec["completion_tokens"]
            STATS["cost_usd"] += rec["cost"]
        return parse_label(out, task)
    except Exception:
        return None

# Unit-test the parser on the exact strings models actually emit. A parser bug here silently
# corrupts every number in the paper, so this assert block is worth its weight.
_cases_bin = [("SARCASTIC", 1), ("NON_SARCASTIC", 0), ("non-sarcastic", 0), ("Label: SARCASTIC", 1),
              ("  sarcastic.", 1), ("NOT SARCASTIC", 0), ("I cannot answer", None)]
for s, want in _cases_bin:
    got = parse_label(s, "ben_sarc_binary")
    assert got == want, f"binary parse {s!r} -> {got}, expected {want}"
_cases_tern = [("NEUTRAL", 1), ("NON_SARCASTIC", 0), ("SARCASTIC", 2), ("neutral", 1)]
for s, want in _cases_tern:
    got = parse_label(s, "banglasarc3_ternary")
    assert got == want, f"ternary parse {s!r} -> {got}, expected {want}"
print("label parser unit tests passed")

## 7 · Cost estimate — read this before the full run

Nothing has been spent yet beyond the one-line auth check.

In [ ]:
n_jobs = 0
for m in MODELS:
    for t in EVAL:
        for s in SETTINGS:
            n_jobs += len(EVAL[t])
cc_jobs = 0
if RUN_CROSS_CORPUS:
    n_pairs = len(CROSS_CORPORA) ** 2
    cc_jobs = len(MODELS) * n_pairs * CROSS_CORPUS_MAX_N

cached_total = sum(len(c) for c in CACHES.values())
print(f"planned in-domain calls : {n_jobs:,}")
print(f"planned cross-corpus    : {cc_jobs:,}")
print(f"TOTAL                   : {n_jobs + cc_jobs:,}")
print(f"already cached          : {cached_total:,}  (these cost nothing to re-run)")
print()
print("Rough cost on cheap models (~$0.15/M in, ~$0.60/M out, ~350 tok/call in, ~5 tok/call out):")
est = (n_jobs + cc_jobs) * (350 * 0.15 + 5 * 0.60) / 1e6
print(f"  ~${est:.2f}   (frontier models are ~10-20x this)")
print()
print("Set MAX_EVAL_N=None for the full test splits when you are ready for the camera-ready run.")

## 8 · Run the in-domain evaluation

Resume-safe: interrupt and re-run at will.

In [ ]:
def run_batch(model, task, setting, eval_df, exemplars):
    """Evaluate one (model, task, setting). Returns a predictions DataFrame.

    Cache hits are resolved serially first, so only genuine misses hit the thread pool — this keeps
    a re-run of a fully-cached config near-instant and network-free.
    """
    toks = tokens_for(task)
    cache = CACHES[model]
    jobs, out = [], {}

    for i, r in eval_df.iterrows():
        msgs = build_messages(task, setting, r["text"], exemplars)
        h = req_hash(model, task, setting, msgs)
        if h in cache:
            out[i] = cache[h]
            with _stats_lock:
                STATS["hits"] += 1
        else:
            jobs.append((i, h, msgs))

    def work(job):
        i, h, msgs = job
        try:
            txt, u = call_llm(model, msgs)
            rec = {"hash": h, "model": model, "task": task, "setting": setting,
                   "raw": txt,
                   "prompt_tokens": int(u.get("prompt_tokens", 0) or 0),
                   "completion_tokens": int(u.get("completion_tokens", 0) or 0),
                   "cost": float(u.get("cost", 0) or 0),
                   "error": None}
        except Exception as e:
            rec = {"hash": h, "model": model, "task": task, "setting": setting,
                   "raw": "", "prompt_tokens": 0, "completion_tokens": 0, "cost": 0.0,
                   "error": str(e)[:300]}
        append_cache(model, rec)
        cache[h] = rec
        with _stats_lock:
            STATS["calls"] += 1
            STATS["prompt_tokens"] += rec["prompt_tokens"]
            STATS["completion_tokens"] += rec["completion_tokens"]
            STATS["cost_usd"] += rec["cost"]
            if rec["error"]:
                STATS["errors"] += 1
        return i, rec

    if jobs:
        done = 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = [ex.submit(work, j) for j in jobs]
            for f in as_completed(futs):
                i, rec = f.result()
                out[i] = rec
                done += 1
                if done % 50 == 0 or done == len(jobs):
                    print(f"    {model.split('/')[-1]:28s} {task:20s} {setting:10s} "
                          f"{done:5d}/{len(jobs):5d}  ${STATS['cost_usd']:.3f}", end="\r")
        print()

    rows = []
    need_norm = []
    for i, r in eval_df.iterrows():
        rec = out[i]
        lab = parse_label(rec.get("raw", ""), task)
        # parse_stage distinguishes a clean regex parse from a normalizer rescue from an outright
        # failure. Reviewers ask how much the fallback machinery is carrying — this answers it.
        rows.append({"text": r["text"], "gold_label": int(r["y"]), "pred_raw": rec.get("raw", ""),
                     "parsed": lab, "parse_stage": "regex" if lab is not None else "failed",
                     "error": rec.get("error")})
        if lab is None:
            need_norm.append(len(rows) - 1)

    # Only unparseable responses reach the normalizer — typically a tiny fraction, so this is cheap.
    if need_norm and USE_NORMALIZER:
        print(f"    normalizing {len(need_norm)} unparsed responses via {NORMALIZER} ...")
        with ThreadPoolExecutor(max_workers=min(4, MAX_WORKERS)) as ex:
            futs = {ex.submit(normalize_label, rows[k]["pred_raw"], task): k for k in need_norm}
            for f in as_completed(futs):
                k = futs[f]
                try:
                    v = f.result()
                    rows[k]["parsed"] = v
                    if v is not None:
                        rows[k]["parse_stage"] = "normalizer"
                except Exception:
                    pass

    df = pd.DataFrame(rows)
    df["parse_ok"] = df["parsed"].notna()          # resolved by EITHER regex or the normalizer
    df["parse_regex_ok"] = df["parse_stage"].eq("regex")   # resolved by regex alone
    df["pred_label"] = df["parsed"]
    if FALLBACK_TO_MAJORITY:
        df["pred_label"] = df["pred_label"].fillna(MAJORITY[task])
    df["pred_label"] = df["pred_label"].astype(int)
    df["correct"] = (df["pred_label"] == df["gold_label"]).astype(int)
    return df

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

t0 = time.time()
RUNS = {}   # (model, task, setting) -> predictions DataFrame

for model in MODELS:
    for task in EVAL:
        ex = pick_exemplars(DATA[task]["train"], task)
        for setting in SETTINGS:
            key = (model, task, setting)
            print(f"\n>> {model} | {task} | {setting}  (n={len(EVAL[task])})")
            df = run_batch(model, task, setting, EVAL[task], ex if setting == "few_shot" else None)
            RUNS[key] = df

            # NB12-compatible schema so these predictions can be dropped straight into the
            # significance notebook if you ever want to re-run it.
            slug = re.sub(r"[^A-Za-z0-9]+", "_", model).strip("_")
            df[["text", "gold_label", "pred_label", "correct", "pred_raw",
                "parse_ok", "parse_stage"]].to_csv(
                LLM_PRED / f"16_llm_{slug}_{setting}_{task}_test_predictions.csv", index=False)

            f1 = f1_score(df["gold_label"], df["pred_label"], average="macro", zero_division=0)
            acc = accuracy_score(df["gold_label"], df["pred_label"])
            print(f"   macro-F1={f1:.4f}  acc={acc:.4f}  regex={df['parse_regex_ok'].mean():.1%}  "
                  f"rescued={(df['parse_stage']=='normalizer').mean():.1%}  "
                  f"unresolved={(df['parse_stage']=='failed').mean():.1%}  "
                  f"errors={df['error'].notna().sum()}")

print(f"\nDONE in {(time.time()-t0)/60:.1f} min | calls={STATS['calls']:,} cache_hits={STATS['hits']:,} "
      f"errors={STATS['errors']} | spend=${STATS['cost_usd']:.3f}")

## 9 · Metrics

Reported per run:
- **macro-F1 (headline)** — parse failures folded into the train-majority class.
- **macro-F1 (parsed only)** — failures excluded, for transparency about how much the fallback carries.
- **bootstrap 95% CI** — same 1000-resample procedure as NB12, so CIs are comparable across notebooks.

In [ ]:
def boot_ci(y, pred, B=BOOTSTRAP_B, seed=SEED):
    # Identical resampling scheme to NB12 so the LLM CIs are directly comparable to the
    # fine-tuned models' CIs already in the paper.
    rng = np.random.default_rng(seed)
    y = np.asarray(y); pred = np.asarray(pred); n = len(y)
    s = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, n)
        s[i] = f1_score(y[idx], pred[idx], average="macro", zero_division=0)
    return float(s.mean()), float(np.percentile(s, 2.5)), float(np.percentile(s, 97.5))

rows = []
for (model, task, setting), df in RUNS.items():
    y, p = df["gold_label"].values, df["pred_label"].values
    bm, lo, hi = boot_ci(y, p)
    sub = df[df["parse_ok"]]
    f1_parsed = (f1_score(sub["gold_label"], sub["pred_label"], average="macro", zero_division=0)
                 if len(sub) > 10 else np.nan)
    rows.append({
        "model": model, "task": task, "setting": setting,
        "n_eval": len(df),
        "macro_f1": round(f1_score(y, p, average="macro", zero_division=0), 4),
        "accuracy": round(accuracy_score(y, p), 4),
        "boot_mean": round(bm, 4), "ci95_lo": round(lo, 4), "ci95_hi": round(hi, 4),
        "macro_f1_parsed_only": round(f1_parsed, 4) if f1_parsed == f1_parsed else "",
        "parse_ok_rate": round(df["parse_ok"].mean(), 4),
        "parse_regex_rate": round(df["parse_regex_ok"].mean(), 4),
        "normalizer_rescue_rate": round((df["parse_stage"] == "normalizer").mean(), 4),
        "unresolved_rate": round((df["parse_stage"] == "failed").mean(), 4),
        "api_error_rate": round(df["error"].notna().mean(), 4),
        "prompt_lang": PROMPT_LANG,
        "fewshot_per_class": FEWSHOT_PER_CLASS if setting == "few_shot" else 0,
    })

llm_df = pd.DataFrame(rows).sort_values(["task", "model", "setting"]).reset_index(drop=True)
llm_df.to_csv(FT / "18_llm_baselines.csv", index=False)
print(llm_df.to_string(index=False))

## 10 · Fine-tuned reference points

Pulled from the prediction CSVs NB05/06/09 already wrote — nothing is retrained.

In [ ]:
def find_pred(stem):
    n = f"{stem}_test_predictions.csv"
    for base in (PRED, OUT / "test_awp" / "predictions"):
        if (base / n).exists():
            return base / n
    for p in ROOT.rglob(n):
        if "finalized_outputs" not in p.parts and "llm_calls" not in p.parts:
            return p
    return None

def pred_metrics(stem):
    p = find_pred(stem)
    if p is None:
        return None
    d = pd.read_csv(p)
    g = d["gold_label"] if "gold_label" in d else d["y_true"]
    y = d["pred_label"] if "pred_label" in d else d["y_pred"]
    return {"stem": stem, "path": p,
            "macro_f1": round(f1_score(g, y, average="macro", zero_division=0), 4),
            "accuracy": round(accuracy_score(g, y), 4), "df": d}

# Per-task fine-tuned reference: prefer the proposed model, fall back through the backbone runs.
FT_STEMS = {
    "ben_sarc_binary":     ["09_FINAL_proposed_ben_sarc_binary", "09b_fgm_awp_ben_sarc_binary",
                            "05_baseline_banglabert_ben_sarc_binary", "06_banglabert_ben_sarc_binary"],
    "banglasarc_binary":   ["06_banglabert_banglasarc_binary", "05_baseline_banglabert_banglasarc_binary"],
    "banglasarc3_binary":  ["06_banglabert_banglasarc3_binary", "05_baseline_banglabert_banglasarc3_binary"],
    "banglasarc3_ternary": ["06_banglabert_banglasarc3_ternary", "05_baseline_banglabert_banglasarc3_ternary"],
}
FT_REF = {}
for task, stems in FT_STEMS.items():
    for s in stems:
        m = pred_metrics(s)
        if m:
            FT_REF[task] = m
            print(f"  {task:22s} <- {s:42s} macro-F1={m['macro_f1']:.4f}")
            break
    else:
        print(f"  {task:22s} <- (no fine-tuned predictions found)")

## 11 · LLM vs fine-tuned + McNemar

McNemar is computed on the **intersection of test items** (matched on `text`), so the comparison is
paired and honest even when `MAX_EVAL_N` subsampled the LLM run.

In [ ]:
from scipy.stats import chi2 as chi2dist

def mcnemar(ca, cb):
    # Continuity-corrected McNemar, same implementation as NB12.
    b = int(((ca == 1) & (cb == 0)).sum())
    c = int(((ca == 0) & (cb == 1)).sum())
    n = b + c
    if n == 0:
        return 0.0, 1.0, b, c
    chi2 = (abs(b - c) - 1) ** 2 / n
    return float(chi2), float(1 - chi2dist.cdf(chi2, 1)), b, c

rows = []
for (model, task, setting), df in RUNS.items():
    ref = FT_REF.get(task)
    if ref is None:
        continue
    rd = ref["df"].dropna(subset=["text"]).drop_duplicates("text").set_index("text")
    ld = df.drop_duplicates("text").set_index("text")
    idx = sorted(set(rd.index) & set(ld.index))
    if len(idx) < 50:
        print(f"  skip {task}/{model}/{setting}: only {len(idx)} shared items")
        continue
    y  = rd.loc[idx, "gold_label"].values
    pa = rd.loc[idx, "pred_label"].values      # fine-tuned
    pb = ld.loc[idx, "pred_label"].values      # LLM
    ca = (pa == y).astype(int); cb = (pb == y).astype(int)
    f_ft  = f1_score(y, pa, average="macro", zero_division=0)
    f_llm = f1_score(y, pb, average="macro", zero_division=0)
    chi2, p, b, c = mcnemar(ca, cb)
    rows.append({"task": task, "llm_model": model, "setting": setting,
                 "finetuned_system": ref["stem"],
                 "f1_finetuned": round(f_ft, 4), "f1_llm": round(f_llm, 4),
                 "delta_ft_minus_llm": round(f_ft - f_llm, 4),
                 "mcnemar_chi2": round(chi2, 4), "p_value": p,
                 "n_pairs": len(idx)})

vs = pd.DataFrame(rows)
if len(vs):
    # Bonferroni across the whole family of comparisons — the same correction NB12 applies.
    vs["p_bonferroni"] = (vs["p_value"] * len(vs)).clip(upper=1.0)
    vs["finetuned_sig_better"] = (vs["p_bonferroni"] < 0.05) & (vs["delta_ft_minus_llm"] > 0)
    vs = vs.sort_values(["task", "llm_model", "setting"]).reset_index(drop=True)
    vs.to_csv(FT / "18_llm_vs_finetuned.csv", index=False)
    print(vs.to_string(index=False, float_format="%.4f"))
else:
    print("no paired comparisons available")

## 12 · Few-shot cross-corpus transfer (optional)

Zero-shot has no source corpus, so a transfer matrix is undefined for it. Few-shot does — the
exemplars *are* the source — so this 3x3 matrix directly parallels NB10, **including the same
leakage blocking**: target-test rows whose `norm_key` appears anywhere in the source corpus are
removed before scoring.

If the LLM's off-diagonal drop is much smaller than the fine-tuned model's, that is a genuinely
interesting result and strengthens the artefact argument — it means the fine-tuned model is
learning corpus-specific cues the LLM never saw.

In [ ]:
CC_ROWS = []
if RUN_CROSS_CORPUS:
    # Binary variants only: the corpora must share a label space for transfer to be meaningful.
    CC_TASK = {"ben_sarc": "ben_sarc_binary", "banglasarc": "banglasarc_binary",
               "banglasarc3": "banglasarc3_binary"}
    avail = [c for c in CROSS_CORPORA if CC_TASK[c] in DATA]
    print("corpora available:", avail)

    for model in MODELS:
        for src in avail:
            src_task = CC_TASK[src]
            ex = pick_exemplars(DATA[src_task]["train"], src_task)
            # Union of ALL source splits, matching NB10's blocking rule exactly.
            src_keys = set(DATA[src_task]["train"]["norm_key"]) | set(DATA[src_task]["test"]["norm_key"])
            for tgt in avail:
                tgt_task = CC_TASK[tgt]
                te = DATA[tgt_task]["test"].copy()
                n_total = len(te)
                n_removed = 0
                if tgt != src:
                    keep = ~te["norm_key"].isin(src_keys)
                    n_removed = int((~keep).sum())
                    te = te[keep]
                if len(te) == 0:
                    continue
                te = subsample(te, CROSS_CORPUS_MAX_N)
                print(f"\n>> CC {model.split('/')[-1]} | {src} -> {tgt}  "
                      f"(n={len(te)}, leak-removed={n_removed})")
                df = run_batch(model, tgt_task, "few_shot", te, ex)
                y, p = df["gold_label"].values, df["pred_label"].values
                CC_ROWS.append({
                    "model": model, "source": src, "target": tgt, "in_domain": src == tgt,
                    "n_eval_total": n_total, "n_removed_leak": n_removed, "n_eval_clean": len(te),
                    "macro_f1": round(f1_score(y, p, average="macro", zero_division=0), 4),
                    "accuracy": round(accuracy_score(y, p), 4),
                    "parse_ok_rate": round(df["parse_ok"].mean(), 4),
                })
                print(f"   macro-F1={CC_ROWS[-1]['macro_f1']:.4f}")

    if CC_ROWS:
        cc = pd.DataFrame(CC_ROWS)
        cc.to_csv(FT / "18_llm_cross_corpus_matrix.csv", index=False)
        for m in cc["model"].unique():
            piv = cc[cc["model"] == m].pivot(index="source", columns="target", values="macro_f1")
            slug = re.sub(r"[^A-Za-z0-9]+", "_", m).strip("_")
            piv.to_csv(FT / f"18_llm_cross_corpus_pivot_{slug}.csv")
            print(f"\n--- {m} ---"); print(piv.to_string())
        # The headline number: worst-case in-domain -> transfer drop, mirroring NB10's framing.
        for m in cc["model"].unique():
            d = cc[cc["model"] == m]
            for s in d["source"].unique():
                idm = d[(d["source"] == s) & (d["in_domain"])]["macro_f1"]
                off = d[(d["source"] == s) & (~d["in_domain"])]["macro_f1"]
                if len(idm) and len(off):
                    print(f"{m.split('/')[-1]:22s} {s:12s} in-domain={idm.iloc[0]:.3f} "
                          f"worst-transfer={off.min():.3f}  drop={idm.iloc[0]-off.min():.3f}")
else:
    print("RUN_CROSS_CORPUS = False — skipped")

## 13 · Figure engine

Byte-for-byte the same style constants as NB15, so these figures sit beside the existing ones with
no visual discontinuity in the paper.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- palette copied verbatim from NB15 so figures are visually consistent across the paper ----
C = {
    "prior":    "#3B6EA5",   # prior-work / reproduced baselines (blue)
    "ours":     "#2E8B6F",   # this work (teal-green)
    "proposed": "#14513E",   # proposed (dark green)
    "reported": "#8C8C8C",   # reference (grey)
    "accent":   "#C77A30",   # highlight (ochre)
    "accent2":  "#A23B3B",   # significant (brick red)
    "nsig":     "#AEB6BD",   # not significant (grey)
    "grid":     "#DDDDDD",
}
SEQ = "viridis"; DIVERGE = "RdYlGn"
IN1, IN2 = 3.5, 7.16   # IEEE single / double column width (inches)

def set_style():
    plt.rcParams.update({
        "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 8.5, "axes.titlesize": 9.5, "axes.titleweight": "bold",
        "axes.labelsize": 8.5, "axes.linewidth": 0.8, "axes.edgecolor": "#333333",
        "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.2,
        "legend.frameon": True, "legend.framealpha": 0.92, "legend.edgecolor": "#CCCCCC",
        "axes.grid": False, "grid.color": C["grid"], "grid.linewidth": 0.6,
        "axes.axisbelow": True, "figure.facecolor": "white", "axes.facecolor": "white",
        "lines.linewidth": 1.4, "patch.linewidth": 0.6,
    })

def _save(fig, outdir, name):
    # PDF for LaTeX (vector, no resampling artefacts), PNG for quick viewing — same as NB15.
    outdir = Path(outdir)
    for ext in ("pdf", "png"):
        fig.savefig(outdir / f"{name}.{ext}")
    plt.close(fig)
    return f"{name}.pdf/.png"

TASK_LABEL = {"ben_sarc_binary": "Ben-Sarc\n(bin)", "banglasarc_binary": "BanglaSarc\n(bin)",
              "banglasarc3_binary": "BanglaSarc3\n(bin)", "banglasarc3_ternary": "BanglaSarc3\n(tern)"}
CORPUS_LABEL = {"ben_sarc": "Ben-Sarc", "banglasarc": "BanglaSarc", "banglasarc3": "BanglaSarc3"}

def short_model(m):
    s = m.split("/")[-1]
    return s.replace("-instruct", "").replace("-chat", "")

set_style()
made, skipped = [], []
def _try(fn, *a, **k):
    try:
        r = fn(*a, **k); made.append(r); print("  ok:", r)
    except Exception as e:
        skipped.append((getattr(fn, "__name__", "?"), str(e))); print("  SKIP", getattr(fn, "__name__", "?"), "->", e)
print("figure engine ready")

In [ ]:
def fig_llm_baselines(df, ft_ref, outdir):
    """Grouped bars: LLM configs per task, with the fine-tuned model as a black reference tick.

    The reference tick (not a bar) keeps the visual hierarchy honest: the fine-tuned model is the
    yardstick, the LLMs are what is being measured against it.
    """
    tasks = [t for t in TASKS if t in df["task"].unique()]
    cfgs = sorted({(r["model"], r["setting"]) for _, r in df.iterrows()},
                  key=lambda x: (x[0], x[1]))
    if not tasks or not cfgs:
        raise ValueError("nothing to plot")

    n = len(cfgs)
    w = 0.8 / n
    x = np.arange(len(tasks))
    fig, ax = plt.subplots(figsize=(IN2, 3.0))

    shades = [C["prior"], C["accent"], C["accent2"], C["ours"], C["nsig"], C["reported"]]
    for j, (m, s) in enumerate(cfgs):
        vals = []
        for t in tasks:
            r = df[(df["task"] == t) & (df["model"] == m) & (df["setting"] == s)]
            vals.append(float(r["macro_f1"].iloc[0]) if len(r) else np.nan)
        pos = x - 0.4 + w * (j + 0.5)
        bars = ax.bar(pos, vals, width=w * 0.92, color=shades[j % len(shades)],
                      edgecolor="#333333", linewidth=0.5,
                      label=f"{short_model(m)} · {s.replace('_', '-')}")
        for b, v in zip(bars, vals):
            if v == v:
                ax.text(b.get_x() + b.get_width() / 2, v + 0.012, f"{v:.2f}",
                        ha="center", va="bottom", fontsize=5.8, rotation=90)

    # Fine-tuned reference ticks
    for i, t in enumerate(tasks):
        if t in ft_ref:
            v = ft_ref[t]["macro_f1"]
            ax.plot([i - 0.42, i + 0.42], [v, v], color="#111111", lw=1.6, zorder=5,
                    label="fine-tuned BanglaBERT" if i == 0 else None)
            ax.text(i + 0.44, v, f"{v:.3f}", fontsize=6, va="center", ha="left", color="#111111")

    ax.set_xticks(x); ax.set_xticklabels([TASK_LABEL.get(t, t) for t in tasks])
    ax.set_ylabel("test macro-F1"); ax.set_ylim(0, 1.08)
    ax.grid(axis="y", alpha=0.5)
    ax.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, 1.30))
    return _save(fig, outdir, "F_llm_baselines")

_try(fig_llm_baselines, llm_df, FT_REF, FF)

In [ ]:
def fig_llm_vs_finetuned(llm_df, ft_ref, outdir, task="ben_sarc_binary"):
    """Horizontal ranking on the headline corpus: where LLMs actually land vs the fitted models.

    Reads the existing grand ranking if NB15 produced it, so the LLM rows slot into the same
    ordering the paper already uses rather than inventing a competing ranking.
    """
    rows = []
    gr = FT / "15_grand_ranking.csv"
    if gr.exists():
        g = pd.read_csv(gr)
        fcol = next((c for c in ("macro_f1", "test_macro_f1", "f1") if c in g.columns), None)
        ncol = next((c for c in ("system", "model", "method", "name") if c in g.columns), None)
        if fcol and ncol:
            for _, r in g.head(6).iterrows():
                rows.append({"label": str(r[ncol])[:42], "v": float(r[fcol]), "kind": "fitted"})
    if not rows and task in ft_ref:
        rows.append({"label": f"Fine-tuned ({ft_ref[task]['stem'][:26]})",
                     "v": ft_ref[task]["macro_f1"], "kind": "fitted"})

    d = llm_df[llm_df["task"] == task]
    for _, r in d.iterrows():
        rows.append({"label": f"{short_model(r['model'])} · {r['setting'].replace('_','-')}",
                     "v": float(r["macro_f1"]), "kind": "llm"})

    if not rows:
        raise ValueError("no rows")
    rd = pd.DataFrame(rows).sort_values("v", ascending=True).reset_index(drop=True)
    cols = [C["proposed"] if k == "fitted" else C["accent"] for k in rd["kind"]]

    fig, ax = plt.subplots(figsize=(IN2, 0.32 * len(rd) + 1.1))
    b = ax.barh(np.arange(len(rd)), rd["v"], color=cols, edgecolor="#333333", linewidth=0.5)
    for i, v in enumerate(rd["v"]):
        ax.text(v + 0.008, i, f"{v:.3f}", va="center", fontsize=7)
    ax.set_yticks(np.arange(len(rd))); ax.set_yticklabels(rd["label"], fontsize=7)
    ax.set_xlabel("test macro-F1"); ax.set_xlim(0, 1.02)
    ax.grid(axis="x", alpha=0.5)

    import matplotlib.patches as mp
    ax.legend(handles=[mp.Patch(color=C["proposed"], label="fine-tuned (this work / prior)"),
                       mp.Patch(color=C["accent"], label="LLM (zero-/few-shot)")],
              loc="lower right")
    ax.set_title(f"{TASK_LABEL.get(task, task).replace(chr(10), ' ')} — fitted models vs LLM prompting")
    return _save(fig, outdir, "F_llm_vs_finetuned")

_try(fig_llm_vs_finetuned, llm_df, FT_REF, FF)

In [ ]:
def fig_llm_cross_corpus(outdir):
    """Transfer heatmap per model, styled like NB15's F_cross_dataset_heatmap for direct comparison."""
    f = FT / "18_llm_cross_corpus_matrix.csv"
    if not f.exists():
        raise FileNotFoundError("no cross-corpus matrix (RUN_CROSS_CORPUS was False)")
    cc = pd.read_csv(f)
    models = list(cc["model"].unique())
    fig, axes = plt.subplots(1, len(models), figsize=(IN2, 2.6 + 0.1 * len(models)), squeeze=False)

    for ax, m in zip(axes[0], models):
        d = cc[cc["model"] == m]
        order = [c for c in CROSS_CORPORA if c in d["source"].unique()]
        piv = d.pivot(index="source", columns="target", values="macro_f1").reindex(
            index=order, columns=order)
        # Same 0.3-1.0 scale as NB15's transfer heatmap so the two are visually comparable.
        im = ax.imshow(piv.values, cmap=DIVERGE, vmin=0.3, vmax=1.0, aspect="auto")
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                v = piv.values[i, j]
                if v == v:
                    ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=7.5,
                            fontweight="bold" if i == j else "normal")
                if i == j:
                    ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False,
                                               edgecolor="black", lw=1.6))
        ax.set_xticks(range(len(order))); ax.set_xticklabels([CORPUS_LABEL[c] for c in order], fontsize=7)
        ax.set_yticks(range(len(order))); ax.set_yticklabels([CORPUS_LABEL[c] for c in order], fontsize=7)
        ax.set_xlabel("evaluated on"); ax.set_ylabel("exemplars from")
        ax.set_title(short_model(m), fontsize=8)
    fig.colorbar(im, ax=axes[0].tolist(), label="macro-F1", fraction=0.035, pad=0.02)
    return _save(fig, outdir, "F_llm_cross_corpus")

if RUN_CROSS_CORPUS and CC_ROWS:
    _try(fig_llm_cross_corpus, FF)
else:
    print("  skip cross-corpus figure (no data)")

## 14 · Provenance, cost report, manifest

In [ ]:
run_cfg = {
    "notebook": "16_llm_baselines",
    "timestamp": pd.Timestamp.now().isoformat(),
    "debug": DEBUG,
    "seed": SEED,
    "models": MODELS,
    "normalizer": NORMALIZER if USE_NORMALIZER else None,
    "base_url": BASE_URL,
    "tasks": list(EVAL.keys()),
    "settings": SETTINGS,
    "prompt_lang": PROMPT_LANG,
    "fewshot_per_class": FEWSHOT_PER_CLASS,
    "max_eval_n": MAX_EVAL_N,
    "eval_n_per_task": {t: int(len(EVAL[t])) for t in EVAL},
    "test_n_per_task": {t: int(len(DATA[t]["test"])) for t in DATA},
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "fallback_to_majority": FALLBACK_TO_MAJORITY,
    "majority_class_per_task": {t: int(c) for t, c in MAJORITY.items()},
    "bootstrap_B": BOOTSTRAP_B,
    "run_cross_corpus": RUN_CROSS_CORPUS,
    "cross_corpus_max_n": CROSS_CORPUS_MAX_N if RUN_CROSS_CORPUS else None,
}
json.dump(run_cfg, open(LLM / "run_config.json", "w"), indent=2, ensure_ascii=False)

cost = dict(STATS)
cost["total_cached_responses"] = sum(len(c) for c in CACHES.values())
json.dump(cost, open(LLM / "cost_report.json", "w"), indent=2)
print("run_config.json + cost_report.json written")
print(json.dumps(cost, indent=2))

In [ ]:
# Additively patch NB15's MANIFEST so the paper build sees the new artefacts. We only ever ADD
# keys and merge lists — nothing NB15 wrote is overwritten or removed.
mf = FINAL / "MANIFEST.json"
if mf.exists():
    man = json.load(open(mf))
    man["generated_figures"] = sorted(set(man.get("generated_figures", [])) | {p.name for p in FF.glob("*.png")})
    man["vector_figures"]    = sorted(set(man.get("vector_figures", []))    | {p.name for p in FF.glob("*.pdf")})
    man["tables"]            = sorted(set(man.get("tables", []))            | {p.name for p in FT.iterdir()})
    man["n_figures"] = len(list(FF.glob("*.png")))
    man["n_tables"]  = len(list(FT.glob("*.csv"))) + len(list(FT.glob("*.json")))
    man["llm_baselines"] = {
        "models": MODELS, "settings": SETTINGS, "prompt_lang": PROMPT_LANG,
        "max_eval_n": MAX_EVAL_N, "figures_skipped": skipped,
        "tables": [t for t in (p.name for p in FT.iterdir()) if t.startswith("18_")],
        "cost_usd": round(STATS["cost_usd"], 4),
    }
    json.dump(man, open(mf, "w"), indent=2)
    print("MANIFEST.json patched (additive — NB15 entries preserved)")
else:
    print("no MANIFEST.json found — skipping patch (run NB15 if you want one)")

## 15 · Summary — the numbers to put in the paper

In [ ]:
print("=" * 92)
print("  NB16 SUMMARY — LLM ZERO-/FEW-SHOT BASELINES")
print("=" * 92)
if DEBUG:
    print("\n  *** DEBUG=True — these are smoke-test numbers on a tiny subset. NOT for the paper. ***")
    print("  *** Set DEBUG=False (and ideally MAX_EVAL_N=None) for the real run.              ***\n")

print(f"\nModels      : {', '.join(MODELS)}")
print(f"Settings    : {', '.join(SETTINGS)} | prompt_lang={PROMPT_LANG} | few-shot={FEWSHOT_PER_CLASS}/class")
print(f"Spend       : ${STATS['cost_usd']:.3f} | calls={STATS['calls']:,} | cache hits={STATS['hits']:,}")

print("\n--- headline: best LLM vs fine-tuned, per task ---")
for t in EVAL:
    d = llm_df[llm_df["task"] == t]
    if not len(d):
        continue
    best = d.loc[d["macro_f1"].idxmax()]
    ftv = FT_REF[t]["macro_f1"] if t in FT_REF else float("nan")
    gap = ftv - best["macro_f1"]
    print(f"  {t:22s} best LLM {best['macro_f1']:.4f} ({short_model(best['model'])}/{best['setting']:9s})"
          f" | fine-tuned {ftv:.4f} | gap {gap:+.4f}")

if len(vs):
    nsig = int(vs["finetuned_sig_better"].sum())
    print(f"\n--- significance: fine-tuned beats LLM in {nsig}/{len(vs)} paired comparisons "
          f"(McNemar, Bonferroni-corrected p<0.05) ---")

if RUN_CROSS_CORPUS and CC_ROWS:
    cc = pd.DataFrame(CC_ROWS)
    print("\n--- cross-corpus (few-shot): in-domain vs worst transfer ---")
    for m in cc["model"].unique():
        d = cc[cc["model"] == m]
        idm = d[d["in_domain"]]["macro_f1"].mean()
        off = d[~d["in_domain"]]["macro_f1"].mean()
        print(f"  {short_model(m):26s} mean in-domain={idm:.3f}  mean transfer={off:.3f}  "
              f"drop={idm-off:.3f}")

print("\n--- written ---")
for p in sorted(FT.glob("18_*")):
    print("  table :", p.relative_to(ROOT))
for p in sorted(FF.glob("F_llm_*")):
    print("  figure:", p.relative_to(ROOT))
print("  calls :", (LLM).relative_to(ROOT), "(cache/, predictions/, run_config.json, cost_report.json)")
if skipped:
    print("\n  figures skipped:", skipped)
print("\nNB16 complete. NB15 outputs untouched — no need to re-run it.")

## What to write in the paper

Add to **Related Work** (2 sentences): note that LLM prompting is now a standard comparator for
sarcasm, and that published benchmarks (SarcasmBench, IEEE TAFFC 2025; MixSarc for Bangla) find
LLMs underperform fine-tuned PLMs — your result should corroborate this.

Add to **Results** (one subsection, one table, one figure): report `18_llm_baselines.csv` as the
table and `F_llm_baselines` as the figure. State the fine-tuned/LLM gap and the McNemar result.

**Framing guidance depending on what you get:**

| Outcome | How to frame it |
|---|---|
| LLM well below fine-tuned (expected) | "Prompting alone does not solve Bangla sarcasm; a task-specific fine-tuned encoder remains necessary." Kills the *"just use GPT-4"* reviewer reflex. |
| LLM close to fine-tuned in-domain | Lean harder on cross-corpus: a model with no in-domain training matching a fitted one implies the fitted one's advantage is largely corpus-specific fitting. **Supports your artefact argument.** |
| LLM transfers flatter than fine-tuned (small off-diagonal drop) | Strongest possible result: the fine-tuned model's collapse is corpus-specific overfitting, and a model that never saw the corpus is more stable. Put this in the abstract. |

All three outcomes strengthen the paper. That asymmetry is exactly why this experiment is worth
running before submission rather than after a reviewer demands it.